In [1]:
import os

In [2]:
cwd= os.getcwd()
BASE_DIR= os.path.join(cwd,"..")

if BASE_DIR not in os.sys.path:
    os.sys.path.append(BASE_DIR)

In [3]:
from src.dataclass.schema import Input, Config, Patient, DiagnosisRecord 
from src.models.classifier import Classifier
from src.database.medical_database import MedicalDatabase 


In [ ]:
"""
patient= Input()

patient.kinh_nguyet= False
patient.da_day= False
patient.tri= False
patient.pregnant= False
patient.man_tinh= True
patient.diet= False
patient.cancer= False
patient.phau_thuat= False


patient.rbc= 4.65
patient.hb= 80
patient.mcv= 56.8
patient.mchc= 303
patient.rdw= 24.4
patient.fe= 4.9
patient.ferritin= 19.4
patient.crp= 2.2
patient.hba= 97.6
patient.hba2= 2.4


config= Config()

c= Classifier(patient, config)
result= c.classify()
print(result.diagnoses)
"""

In [ ]:
info = Patient(id= 0, full_name="Bi",
                 dob= "2003-11-24", gender= "Nam", phone_number= "10x", address= "Tây Hồ, Hà Nội")

record= DiagnosisRecord(
    id= 2,
    patient_id= 0,
    rbc= 4.61,
    hb= 99,
    mcv= 70.9,
    mchc= 303,
    rdw= 16,
    fe= 3.6,
    ferritin= 4.9,
    hba= 72.5,
    hba2= 3.2,
    hbe= 23.9,
    diagnoses= result.diagnoses,
    reasons= result.reasons
)


db= MedicalDatabase()



In [ ]:
db.create_tables()


In [ ]:
db.add_patient(info)


In [ ]:
db.add_record(record)

In [ ]:
db.search_by_field("address", "Tây Hồ")

#### Try with hospital data

In [4]:
import pandas as pd
import os

cwd= os.getcwd()
BASE_DIR= os.path.join(cwd,"..")

data= os.path.join(BASE_DIR, "data/concat_for_eda.csv")
df= pd.read_csv(data)

config= Config()

df['guideline']= None

for idx, r in df.iterrows():
    patient= Input(
        rbc= r['RBC'],
        hb= r['Hb'],
        mcv= r['MCV'],
        mchc= r['MCHC'],
        rdw= r['RDW-CV'],
        fe= r['Fe'],
        ferritin= r['Ferritin'],
        transferrin= r['Transferin'],
        tibc= r['TIBC'],
        crp= r['CRP'],
        ret_he= r['Ret-He'],
        hba= r['HbA1'],
        hba2= r['HbA2'],
        hbf= r['HbF'],
        hbh= r['HbH'],
        hbbart= r['HbBart'],
        hbs= r['HbS'],
        hbe= r['HbE'],
        hb_other= r['Hb khác'],
        dotbiengen= r['Đột biến gen thalassemia'],
        tsat= r['TSAT (%)'],
        mch= r['MCH'],
        man_tinh= r['Tiền sử hoặc bệnh kèm theo']
        )
    c= Classifier(patient, config)
    result= c.classify()
    
    df.loc[idx, "guideline"]= result.diagnoses

df.to_csv("result.csv", index= False)
    

In [11]:
import pandas as pd
import matplotlib.pyplot as plt

# đếm số lượng True của từng cột
counts = df[["ACD", "IDA", "Alpha thalassemia", "Beta thalassemia"]].sum()

print(counts)

# vẽ bar chart
plt.figure(figsize=(10, 8))

plt.bar(counts.index, counts.values)

# ghi số lên cột
for i, v in enumerate(counts.values):
    plt.text(i, v, str(int(v)), ha="center", va="bottom")

plt.xlabel("Disease")
plt.ylabel("Count")
plt.title("Number of Cases")
plt.xticks(rotation=30)
plt.savefig("disease_counts.png")
plt.show()

ACD                  42
IDA                  35
Alpha thalassemia    11
Beta thalassemia     10
dtype: int64


/tmp/ipykernel_12237/2151043839.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
from sklearn.metrics import classification_report

import pandas as pd
import os 

cwd= os.getcwd()
BASE_DIR= os.path.join(cwd,"..")

data= os.path.join(BASE_DIR, "notebooks/result.csv")
df= pd.read_csv(data)

df = df[~pd.isna(df["guideline"])]
# y_true = guideline thật
y_true = df[["ACD", "IDA", "Alpha thalassemia", "Beta thalassemia"]]

# y_pred từ cột Chẩn đoán
y_pred = pd.DataFrame({
    "ACD": df["guideline"].str.contains("ACD", na=False),
    "IDA": df["guideline"].str.contains("IDA", na=False),
    "Alpha thalassemia": df["guideline"].str.contains("Alpha thalassemia", na=False),
    "Beta thalassemia": df["guideline"].str.contains("Beta thalassemia", na=False),
})

# classification report
report = classification_report(
    y_true,
    y_pred,
    target_names=["ACD", "IDA", "Alpha thalassemia", "Beta thalassemia"]
)

print(report)

                   precision    recall  f1-score   support

              ACD       1.00      0.95      0.98        42
              IDA       0.97      0.86      0.91        35
Alpha thalassemia       1.00      0.91      0.95        11
 Beta thalassemia       0.90      0.90      0.90        10

        micro avg       0.98      0.91      0.94        98
        macro avg       0.97      0.90      0.93        98
     weighted avg       0.98      0.91      0.94        98
      samples avg       0.98      0.94      0.95        98



In [7]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_true, y_pred)

print("Accuracy:", acc)


Accuracy: 0.9010989010989011


In [8]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import pandas as pd


# convert prediction -> label chuẩn
def pred_label(x):    
    x = str(x).lower()

    labels = []

    if "acd" in x:
        labels.append("ACD")

    if "ida" in x:
        labels.append("IDA")

    elif "beta thalassemia" in x:
        labels.append("Beta thalassemia")

    elif "alpha thalassemia" in x:
        labels.append("Alpha thalassemia")
    
    
    if len(labels) == 0:
        return "None"

    return ", ".join(sorted(labels))

# tạo label
y_true = df["Chẩn đoán"]
y_pred = df["guideline"].apply(pred_label)

# labels xuất hiện
all_labels = sorted(list(set(y_true) | set(y_pred)))

# confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=all_labels)

# plot
plt.figure(figsize=(10, 8))

plt.imshow(cm, cmap= 'coolwarm')

plt.xticks(range(len(all_labels)), all_labels, rotation=45)
plt.yticks(range(len(all_labels)), all_labels)

# ghi số
for i in range(len(all_labels)):
    for j in range(len(all_labels)):
        plt.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center"
        )

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")

plt.colorbar()

plt.tight_layout()
plt.savefig("confusion.png")

plt.show()

/tmp/ipykernel_12237/4094618753.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
